In [1]:
from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir

logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)

In [ ]:
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [30]:
fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV9_T2-001.tsm'
fr = 640
print(fname, fr)
assert(os.path.exists(fname), 'File not in the correct place')

C:\Users\ICNLab\caiman_data\testdata\testdata\FOV9_T2-001.tsm 640


<>:4: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:4: SyntaxWarning: assertion is always true, perhaps remove parentheses?
C:\Users\ICNLab\AppData\Local\Temp\ipykernel_23700\4058526029.py:4: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(os.path.exists(fname), 'File not in the correct place')


In [4]:
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

In [5]:
m_orig = cm.load(fname)
ds_ratio = 0.2


In [6]:

moviehandle = m_orig.resize(1, 1, ds_ratio)
moviehandle.play(q_max=99.5, fr=40, magnification=1)

KeyboardInterrupt: 

In [7]:
try:
    c, dview, n_processes = cm.cluster.setup_cluster(
            backend='local', n_processes=None, single_thread=False)
except:
    dview.close()

mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=True)

In [8]:
m_orig = cm.load(fname)
m_rig = cm.load(mc.mmap_file)
ds_ratio = 0.2
moviehandle = cm.concatenate([m_orig.resize(1, 1, ds_ratio) - mc.min_mov*mc.nonneg_movie,
                              m_rig.resize(1, 1, ds_ratio)], axis=2)
moviehandle.play(fr=60, q_max = 99.5, magnification=1)

100%|██████████| 1/1 [00:08<00:00,  8.64s/it]


KeyboardInterrupt: 

In [9]:
border_to_0 = 0 if mc.border_nan == 'copy' else mc.border_to_0
fname_new = cm.save_memmap_join(mc.mmap_file, base_name='memmap_', add_to_mov=border_to_0, dview=dview)
img = mean_image(mc.mmap_file[0], window=1000, dview=dview)
img = (img-np.mean(img))/np.std(img)
gaussian_blur = False
cn = local_correlations_movie_offline(mc.mmap_file[0], fr=fr, window=fr*4, stride=fr*4, winSize_baseline=fr*2, remove_baseline=True,
                                      gaussian_blur = gaussian_blur, dview=dview).max(axis=0)
img_corr = (cn - np.mean(cn))/np.std(cn)
summary_images = np.stack([img, img, img_corr], axis=0).astype(np.float32)
cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')

In [31]:
#load in my own FOV images
import imageio.v3 as iio
import numpy as np

# Load PNG as RGB (H, W, 3)
img = iio.imread("C:/Users/ICNLab/caiman_data/testdata/testdata/FOV9_T2-001.png").astype(np.float32)

# Extract channels
R = img[:, :, 0]
G = img[:, :, 1]
B = img[:, :, 2]

# Stack to shape (3, H, W)
summary_images = np.stack([R, G, B], axis=0).astype(np.float32)


In [32]:
plt.imshow(summary_images[0], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)

In [33]:
plt.imshow(summary_images[2], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)
img = summary_images.transpose([1, 2, 0])


print(fname[:-4]+'_corr.tif')
height, width = img.shape[:2]
print(height, width)




C:\Users\ICNLab\caiman_data\testdata\testdata\FOV9_T2-001_corr.tif
512 512


In [28]:
import scipy.io as sio
import numpy as np

# img must already be shape (512, 512, 3)
# and should be float64 (MATLAB double)
img_mat = img.astype(np.float64)

sio.savemat(fname[:-4] + '_summary_images.mat', {'img': img_mat})


In [34]:
weights_path="C:/Users/ICNLab/caiman_data/testdata/testdata/mask_rcnn_neuron_0012.h5"
#download_model('mask_rcnn')
#ROIs, r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
ROIs = r['masks'].transpose([2, 0, 1])
cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')


Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_class_loss': 1.0, 'mrcnn_bbox_loss': 1.0, 'mrcnn_mask_loss': 1.0}
MASK_POOL_SIZE                 14
MASK_SHAPE                  

c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


MADE FIGURE


In [35]:
fig, axs = plt.subplots(1, 2)
axs[0].imshow(summary_images[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')
plt.savefig(fname[:-6] + 'newmrcnn_ROIs.png', format='png', bbox_inches='tight', pad_inches=0)

In [36]:
cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)

In [37]:
ROIs = ROIs                                   # region of interests
index = list(range(len(ROIs)))                # index of neurons
weights = None                                # if None, use ROIs for initialization; to reuse weights check reuse weights block

template_size = 0.02                          # half size of the window length for spike templates, default is 20 ms
context_size = 35                             # number of pixels surrounding the ROI to censor from the background PCA
visualize_ROI = False                         # whether to visualize the region of interest inside the context region
hp_freq_pb = 1 / 3                            # parameter for high-pass filter to remove photobleaching
clip = 100                                    # maximum number of spikes to form spike template
threshold_method = 'adaptive_threshold'       # adaptive_threshold or simple
min_spikes= 10                                # minimal spikes to be found
pnorm = 0.5                                   # a variable deciding the amount of spikes chosen for adaptive threshold method
threshold = 2                                 # threshold for finding spikes only used in simple threshold method, Increase the threshold to find less spikes
do_plot = False                               # plot detail of spikes, template for the last iteration
ridge_bg= 0.05                                # ridge regression regularizer strength for background removement, larger value specifies stronger regularization
sub_freq = 20                                 # frequency for subthreshold extraction
weight_update = 'ridge'                       # ridge or NMF for weight update
n_iter = 2                                    # number of iterations alternating between estimating spike times and spatial filters

opts_dict={'fnames': fname_new,
           'ROIs': ROIs,
           'index': index,
           'weights': weights,
           'template_size': template_size,
           'context_size': context_size,
           'visualize_ROI': visualize_ROI,
           'hp_freq_pb': hp_freq_pb,
           'clip': clip,
           'threshold_method': threshold_method,
           'min_spikes':min_spikes,
           'pnorm': pnorm,
           'threshold': threshold,
           'do_plot':do_plot,
           'ridge_bg':ridge_bg,
           'sub_freq': sub_freq,
           'weight_update': weight_update,
           'n_iter': n_iter}

opts.change_params(params_dict=opts_dict);

In [38]:
vpy = VOLPY(n_processes=n_processes, dview=dview, params=opts)
vpy.fit(n_processes=n_processes, dview=dview)

In [ ]:
# Visualize spatial footprints and traces
print(np.where(vpy.estimates['locality'])[0])    # neurons that pass locality test
idx = np.where(vpy.estimates['locality'] > 0)[0]
utils.view_components(vpy.estimates, img_corr, idx)

[  7   9  20  27  41  50  54  56  59  60  63  72  75  76  84  94 101 103]
Component:0


Component:0
Component:1
Component:1
Component:1
Component:1
Component:2
Component:2
Component:2
Component:3
Component:3
Component:4
Component:4
Component:4
Component:4
Component:4
Component:4
Component:4
Component:4
Component:4
Component:4
Component:4
Component:4
Component:3
Component:3
Component:3
Component:3
Component:7
Component:7
Component:9
Component:11
Component:12
Component:14
Component:15
Component:16
Component:17
Component:10
Component:8
Component:7
Component:6
Component:8
Component:8
Component:8
Component:7
Component:7
Component:7
Component:7
Component:7
Component:7
Component:7
Component:7
Component:7
Component:8
Component:9
Component:9
Component:10
Component:11
Component:12
Component:13
Component:13
Component:14
Component:15
Component:10
Component:9
Component:8
Component:9


In [ ]:
# Reconstructed movie
flip_signal = True    
mv_all = utils.reconstructed_movie(vpy.estimates.copy(), fnames=mc.mmap_file,
                                           idx=idx, scope=(0,1000), flip_signal=flip_signal)
#mv_all.play(fr=40, magnification=3)

100%|██████████| 1/1 [00:11<00:00, 11.39s/it]


In [ ]:
vpy.estimates['ROIs'] = ROIs
save_name = fname[:-4]+'new_volpy'
np.save(save_name, vpy.estimates)

cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)